# Deploy Dialog-RSN-1 from AWS Marketplace

This notebook deploys [Dialog-RSN-1](https://poly.ai), PolyAI's audio-native language model for
customer-service voice agents, from its AWS Marketplace model package to a SageMaker endpoint in
your account, then holds three conversations with it using the stock `openai` Python SDK.

Dialog-RSN-1 listens to caller audio directly, decides when the caller has finished speaking, and
answers with text or a tool call in one pass. It does not produce audio; you pair it with the
text-to-speech engine of your choice. The endpoint speaks the OpenAI Realtime wire protocol, so an
existing OpenAI Realtime client works against it by changing only the base URL.

> **Note:** this is a reference notebook. It needs the model package ARN from your subscription,
> and it creates a GPU endpoint that bills per hour until section 5 deletes it.

## Prerequisites

1. An AWS account subscribed to Dialog-RSN-1 on AWS Marketplace, or an IAM role allowed to
   subscribe (`aws-marketplace:Subscribe`, `aws-marketplace:ViewSubscriptions`,
   `aws-marketplace:Unsubscribe`).
2. An IAM role with **AmazonSageMakerFullAccess**. On SageMaker Studio or a notebook instance
   that is the execution role. On a laptop, set `SAGEMAKER_ROLE_ARN` in the environment.
3. A service quota of at least 1 for `ml.p4d.24xlarge` (or `ml.p5.48xlarge`) **for endpoint
   usage** in your region. A fresh account has 0, and the failure arrives several minutes into
   endpoint creation rather than immediately.
4. Node.js 20 or newer on the machine running this notebook (`node --version`). Section 3
   explains why.
5. The Python packages in `requirements.txt`: `pip install -r requirements.txt`.

## Contents

1. [Subscribe to the model package](#1.-Subscribe-to-the-model-package)
2. [Create an endpoint](#2.-Create-an-endpoint)
3. [Start the bridge](#3.-Start-the-bridge)
4. [Talk to the model with the OpenAI SDK](#4.-Talk-to-the-model-with-the-OpenAI-SDK)
   1. [A first conversation](#A.-A-first-conversation)
   2. [Function calling](#B.-Function-calling)
   3. [Answering from a knowledge base](#C.-Answering-from-a-knowledge-base)
   4. [Text input, and what is on the wire](#D.-Text-input,-and-what-is-on-the-wire)
5. [Delete the endpoint](#5.-Delete-the-endpoint)
6. [Unsubscribe from the listing (optional)](#6.-Unsubscribe-from-the-listing-(optional))
7. [Going further](#7.-Going-further)

Run the cells one at a time with Shift+Enter.

## 1. Subscribe to the model package

1. Open the Dialog-RSN-1 listing on AWS Marketplace.
2. Choose **Continue to subscribe**.
3. On **Subscribe to this software**, review the EULA, pricing and support terms, then choose
   **Accept Offer**.
4. Choose **Continue to configuration** and pick your region. The page shows a **Product Arn**.
   That is the model package ARN. Paste it below.

In [ ]:
model_package_arn = "<model package ARN for your region, from the listing's configuration page>"

In [ ]:
import asyncio
import base64
import contextlib
import json
import os
import socket
import subprocess
import time
import wave
from pathlib import Path

import boto3

region = boto3.session.Session().region_name or os.environ.get("AWS_REGION", "us-east-2")
sm = boto3.client("sagemaker", region_name=region)


def execution_role() -> str:
    # On SageMaker Studio or a notebook instance this is the role the kernel runs as. Elsewhere
    # there is no such thing, so the role has to be named explicitly.
    try:
        from sagemaker import get_execution_role  # noqa: PLC0415

        return get_execution_role()
    except Exception:
        role = os.environ.get("SAGEMAKER_ROLE_ARN")
        if not role:
            raise RuntimeError("Set SAGEMAKER_ROLE_ARN to the execution role for the endpoint") from None
        return role


role = execution_role()
print(f"region {region}\nrole   {role}")

## 2. Create an endpoint

Three API calls: a **model** that points at the package, an **endpoint config** that pairs it
with an instance type, and the **endpoint** itself. Plain `boto3`, so every setting is visible.

Three of those settings matter for this model and are easy to get wrong:

- **Instance type.** Pick one from the listing's recommended list. Dialog-RSN-1 runs one model
  replica per GPU, so an 8-GPU node serves eight times the concurrent calls of a single GPU.
- **`InferenceAmiVersion`.** The container is built on CUDA 12.9. The default inference AMI
  ships an older NVIDIA driver and refuses to start the container at all, with
  `CannotStartContainerError` and no container logs. `al2023-ami-sagemaker-inference-gpu-4-1`
  ships a driver that works.
- **Both timeouts at their 3600 s ceiling.** The image carries the model weights and compiles
  the model per GPU on startup, which takes longer than the defaults allow.

The model and endpoint config are immutable in SageMaker, so their names carry a timestamp. The
endpoint name is the one you will use later.

In [ ]:
endpoint_name = "dialog-rsn-1"
instance_type = "ml.p4d.24xlarge"  # or "ml.p5.48xlarge"

model_name = f"{endpoint_name}-{time.strftime('%Y%m%d-%H%M')}"

sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={"ModelPackageName": model_package_arn},
    # Marketplace packages run network isolated: nothing dials out of the container, and no data
    # leaves your VPC.
    EnableNetworkIsolation=True,
)

sm.create_endpoint_config(
    EndpointConfigName=model_name,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": model_name,
            "InitialInstanceCount": 1,
            "InstanceType": instance_type,
            "InitialVariantWeight": 1.0,
            "InferenceAmiVersion": "al2023-ami-sagemaker-inference-gpu-4-1",
            "ModelDataDownloadTimeoutInSeconds": 3600,
            "ContainerStartupHealthCheckTimeoutInSeconds": 3600,
        }
    ],
)

sm.create_endpoint(EndpointName=endpoint_name, EndpointConfigName=model_name)
print(f"creating {endpoint_name} on {instance_type} from {model_name}")

Expect around 15 minutes: the image is large, and the model loads and compiles on each GPU. The
waiter below polls until the endpoint is `InService` or has failed.

From `InService` onwards the endpoint bills per hour, connected or not, until section 5 deletes it.

In [ ]:
sm.get_waiter("endpoint_in_service").wait(
    EndpointName=endpoint_name, WaiterConfig={"Delay": 30, "MaxAttempts": 120}
)
described = sm.describe_endpoint(EndpointName=endpoint_name)
print(described["EndpointStatus"], described.get("FailureReason", ""))

## 3. Start the bridge

SageMaker gives you no `wss://` URL. A bidirectional-streaming endpoint is reached by calling
`InvokeEndpointWithBidirectionalStream` over HTTP/2 with SigV4 signing. The AWS SDKs do that. An
OpenAI Realtime client opens a WebSocket.

`bridge/bridge.mjs` closes that gap. It listens on `ws://127.0.0.1:8079`, opens one SageMaker
stream per WebSocket connection, and copies frames across unchanged in both directions. Your
client sees a WebSocket; the model sees the stream it expects. Credentials come from the standard
AWS credential chain, the same one `boto3` used above.

It is a Node process because the SageMaker runtime serves HTTP/2 without advertising it via ALPN,
and Node connects by prior knowledge. The Python AWS SDK cannot. You do not have to use it:
a client that calls `InvokeEndpointWithBidirectionalStream` directly with the AWS JavaScript SDK
and carries the same JSON events in `PayloadPart` works too. The bridge is what lets you keep the
client you already have.

In [ ]:
BRIDGE_PORT = 8079
bridge_dir = Path("bridge").resolve()

subprocess.run(["npm", "install", "--no-audit", "--no-fund"], cwd=bridge_dir, check=True, capture_output=True)

bridge = subprocess.Popen(
    ["node", "bridge.mjs"],
    cwd=bridge_dir,
    env={**os.environ, "SAGEMAKER_ENDPOINT": endpoint_name, "AWS_REGION": region, "PORT": str(BRIDGE_PORT)},
    stderr=subprocess.PIPE,
    text=True,
)
print(bridge.stderr.readline().strip())  # bridge: ws://127.0.0.1:8079 -> dialog-rsn-1

for _ in range(50):
    if bridge.poll() is not None:
        raise RuntimeError(f"bridge exited with {bridge.returncode}: {bridge.stderr.read()}")
    with socket.socket() as probe:
        if probe.connect_ex(("127.0.0.1", BRIDGE_PORT)) == 0:
            break
    time.sleep(0.1)
else:
    raise RuntimeError("bridge did not start listening")

## 4. Talk to the model with the OpenAI SDK

Everything from here on is a stock OpenAI Realtime client. The only Dialog-RSN-1-specific line is
the base URL, which points at the bridge. The API key is a placeholder: authentication already
happened on the SigV4 call the bridge made.

A few things about the session, all of which are on the wire in `session.update`:

- Audio goes in as PCM16 mono. The sample files are 16 kHz, so the session says so. 24 kHz, the
  OpenAI default, is also accepted.
- `output_modalities` is `["text"]`. This model does not produce audio.
- Turn taking is the server's. `server_vad` is the only detector and its thresholds are not
  tunable, because the model judges when a turn has ended and the detector only proposes a
  boundary. The one knob is `create_response`. This notebook sets it `false` and asks for each reply explicitly,
  which is what a scripted client does. A live voice client leaves it at the default, `true`, and
  the reply arrives as soon as the model decides the caller has finished.
- Events are sent as plain dicts rather than the SDK's typed helpers, so that PolyAI's extension
  fields, which the SDK does not know about, reach the wire.

In [ ]:
from openai import AsyncOpenAI

BRIDGE_URL = f"ws://127.0.0.1:{BRIDGE_PORT}"  # the SDK appends /realtime itself
SAMPLE_RATE = 16_000
CHUNK_MS = 20  # what a browser worklet or a telephony frame delivers

client = AsyncOpenAI(api_key="not-required", websocket_base_url=BRIDGE_URL)


def session_payload(instructions: str, tools: list | None = None, knowledge: list | None = None) -> dict:
    session = {
        "type": "realtime",
        "model": "dialog-rsn-1",
        "output_modalities": ["text"],
        "instructions": instructions,
        "audio": {
            "input": {
                "format": {"type": "audio/pcm", "rate": SAMPLE_RATE},
                "turn_detection": {"type": "server_vad", "create_response": False},
            }
        },
        "tools": tools or [],
    }
    if knowledge:
        # A PolyAI extension: topics the model may answer from, cited by name on response.done.
        session["poly_knowledge"] = knowledge
    return session


def load_pcm16(path: str) -> bytes:
    with wave.open(path) as wav:
        shape = (wav.getnchannels(), wav.getsampwidth(), wav.getframerate())
        assert shape == (1, 2, SAMPLE_RATE), f"{path}: expected mono 16-bit {SAMPLE_RATE} Hz, got {shape}"
        return wav.readframes(wav.getnframes())


def render(event: dict) -> None:
    # One line per event worth reading. Deltas and bookkeeping events are kept in
    # Conversation.events but not printed.
    kind = event.get("type", "")
    if kind == "input_audio_buffer.speech_started":
        print(f"  [caller speaking from {event.get('audio_start_ms')} ms]")
    elif kind == "input_audio_buffer.speech_stopped":
        print(f"  [caller stopped at {event.get('audio_end_ms')} ms]")
    elif kind == "conversation.item.input_audio_transcription.completed":
        print(f"  caller: {event.get('transcript')!r}")
    elif kind == "response.function_call_arguments.done":
        print(f"  tool call: {event.get('name')}({event.get('arguments')})")
    elif kind == "response.output_text.done" and event.get("text"):
        # A turn that ends in a tool call carries an empty text part before the call; skip it.
        print(f"  agent: {event.get('text')!r}")
    elif kind == "response.done":
        response = event.get("response", {})
        extras = {k: response[k] for k in ("poly_cited_topics", "poly_out_of_domain") if k in response}
        print(f"  [response {response.get('status')}] {extras if extras else ''}")
    elif kind == "error":
        print(f"  ERROR {event.get('error')}")

In [ ]:
class Conversation:
    """One session: sends events, prints what comes back, and answers the model's tool calls."""

    def __init__(self, connection, tool_handlers: dict | None = None) -> None:
        self.connection = connection
        self.tool_handlers = tool_handlers or {}
        self.events: list[dict] = []  # every server event, raw, in order
        self.inbox: asyncio.Queue = asyncio.Queue()
        self.reader = asyncio.create_task(self._read())

    async def _read(self) -> None:
        # Drain continuously, so an event that arrives while we are busy sending audio is still
        # received in order and nothing backs up on the socket.
        try:
            while True:
                await self.inbox.put(json.loads(await self.connection.recv_bytes()))
        except Exception as exc:  # the connection closed, cleanly or otherwise
            await self.inbox.put({"type": "_closed", "reason": repr(exc)})

    async def send(self, event: dict) -> None:
        await self.connection.send(event)

    async def send_audio(self, pcm16: bytes, realtime: bool = True) -> None:
        chunk_bytes = SAMPLE_RATE * CHUNK_MS // 1000 * 2
        started = time.monotonic()
        for i, start in enumerate(range(0, len(pcm16), chunk_bytes)):
            chunk = pcm16[start : start + chunk_bytes]
            await self.send({"type": "input_audio_buffer.append", "audio": base64.b64encode(chunk).decode("ascii")})
            if realtime:  # pace to wall clock, because turn detection is timing dependent
                await asyncio.sleep(max(0.0, started + (i + 1) * CHUNK_MS / 1000 - time.monotonic()))

    async def wait_for(self, event_type: str, timeout_s: float = 45.0) -> dict:
        deadline = time.monotonic() + timeout_s
        while True:
            event = await asyncio.wait_for(self.inbox.get(), timeout=max(0.0, deadline - time.monotonic()))
            if event["type"] == "_closed":
                raise ConnectionError(f"connection closed: {event['reason']}")
            self.events.append(event)
            render(event)
            if event["type"] == event_type:
                return event

    async def turn(self, wav: str | None = None, text: str | None = None, silence_s: float = 1.0) -> dict:
        """One caller turn: speak or type, fall silent, ask for the reply, answer any tool call."""
        if wav:
            print(f"caller plays {wav}")
            await self.send_audio(load_pcm16(wav))
            # Trailing silence is what server-side turn detection needs to see to call the turn over.
            await self.send_audio(b"\x00\x00" * int(SAMPLE_RATE * silence_s))
        if text:
            print(f"caller types {text!r}")
            await self.send(
                {
                    "type": "conversation.item.create",
                    "item": {"type": "message", "role": "user", "content": [{"type": "input_text", "text": text}]},
                }
            )
        await self.send({"type": "response.create"})
        done = await self.wait_for("response.done")

        # A turn that ended in a tool call is not over: run the tool, return the result, and the
        # reply the caller actually hears comes from the generation after it.
        call = next((item for item in done["response"].get("output", []) if item.get("type") == "function_call"), None)
        if call is None:
            return done
        handler = self.tool_handlers.get(call["name"], lambda **_: {"ok": True})
        result = handler(**json.loads(call.get("arguments") or "{}"))
        await self.send(
            {
                "type": "conversation.item.create",
                "item": {"type": "function_call_output", "call_id": call["call_id"], "output": json.dumps(result)},
            }
        )
        await self.send({"type": "response.create"})
        return await self.wait_for("response.done")

    async def close(self) -> None:
        self.reader.cancel()


@contextlib.asynccontextmanager
async def conversation(instructions: str, tools: list | None = None, knowledge: list | None = None, tool_handlers: dict | None = None):
    async with client.realtime.connect(model="dialog-rsn-1") as connection:
        conv = Conversation(connection, tool_handlers)
        try:
            await conv.send({"type": "session.update", "session": session_payload(instructions, tools, knowledge)})
            await conv.wait_for("session.updated", timeout_s=10)
            yield conv
        finally:
            await conv.close()

### A. A first conversation

Two caller turns, answered from a couple of facts in the instructions. The second turn only
makes sense given the first, which shows the server holds the conversation history. Each turn
streams a WAV file in 20 ms chunks at real-time pace, then a second of silence, then asks for
the reply.

Watch the order of what comes back: the turn boundary the model decided on, the reply, and then
the transcript of what the caller said. The transcript arrives after the reply on purpose, so your
application can start speaking without waiting for it.

In [ ]:
RESTAURANT = (
    "You are the assistant for Maison Lumiere, a restaurant. Answer in one short sentence, from these "
    "facts only: we are open until 11 pm every day, and the nearest parking is the public car park on "
    "Bridge Street, two minutes' walk away."
)

async with conversation(RESTAURANT) as conv:
    await conv.turn(wav="audio/opening_hours.wav")
    await conv.turn(wav="audio/followup_parking.wav")

### B. Function calling

Tools are declared the OpenAI way: a name, a description, and a JSON schema for the arguments.
The model calls at most one tool per turn, with complete arguments, and never mixes a tool call
with spoken text in the same turn. Your application runs the tool, returns the result as a
`function_call_output` item, and asks for the follow-up generation.

Here the caller asks to book a table, the model calls `check_availability`, our handler answers,
and the model reports back. The second turn amends the booking; a server that had lost the first
result would re-ask for details the caller already gave. The third turn asks for a human.

In [ ]:
BOOKINGS = '''
You are Sam, a voice assistant for Maison Lumiere, a restaurant. At each turn do exactly one of
(1) answer from what you know, or (2) call a function when the rules say to. Never do both. Keep
spoken answers to one short sentence.

BOOKING FLOW:
- If the user asks to make, change or cancel a booking, call check_availability with the party
  size and time before saying anything about whether it is possible.
- Once you have the availability result, tell the user in one sentence.

TRANSFER BEHAVIOUR:
- If the user asks for a human, an agent, an operator or a real person, call handoff with
  handoff_reason='SPEAK_TO'.
- If the user asks about something unrelated to the restaurant, call handoff with
  handoff_reason='OUT_OF_SCOPE'.
'''

BOOKING_TOOLS = [
    {
        "type": "function",
        "name": "check_availability",
        "description": "Check whether a table is free at a given time. Call before answering any question about booking a table.",
        "parameters": {
            "type": "object",
            "properties": {
                "party_size": {"type": "integer", "description": "Number of people"},
                "date": {"type": "string", "description": "Date in natural language as the caller said it"},
                "time": {"type": "string", "description": "Time in 24-hour HH:MM"},
            },
            "required": ["party_size", "time"],
        },
    },
    {
        "type": "function",
        "name": "handoff",
        "description": "Transfer the caller to a human agent.",
        "parameters": {
            "type": "object",
            "properties": {
                "handoff_reason": {"type": "string", "enum": ["SPEAK_TO", "OUT_OF_SCOPE", "OOD_LOOP", "CALL_BACK"]},
                "handoff_utterance": {"type": "string"},
            },
            "required": ["handoff_reason"],
        },
    },
]

# Stand-ins for your booking system and your telephony platform.
BOOKING_HANDLERS = {
    "check_availability": lambda party_size, time, **_: {"available": True, "party_size": party_size, "time": time, "table": "window"},
    "handoff": lambda handoff_reason, **_: {"status": "transferred", "reason": handoff_reason},
}

async with conversation(BOOKINGS, tools=BOOKING_TOOLS, tool_handlers=BOOKING_HANDLERS) as conv:
    await conv.turn(wav="audio/booking_request.wav")
    await conv.turn(wav="audio/followup_change_size.wav")
    await conv.turn(wav="audio/speak_to_human.wav")

### C. Answering from a knowledge base

`poly_knowledge` is a PolyAI extension on the session: a list of topics, each with a name and the
content the model may answer from. The protocol has no field for this, so it rides on
`session.update` as an extra key, which is why events are sent as plain dicts. A stock OpenAI
client would drop it.

When the model answers from a topic, `response.done` carries `poly_cited_topics` naming it. Every
`response.done` also carries `poly_out_of_domain`, the model's judgement of whether the request
was within this deployment's scope. The last turn here asks about something unrelated, so watch
that flag flip.

The same topic list can travel on a `function_call_output` item instead, which is what a
knowledge-lookup tool returns: your application owns the search, and the passages come back with
the result.

In [ ]:
KNOWLEDGE_INSTRUCTIONS = '''
You are Sam, a voice assistant for Maison Lumiere, a restaurant. Answer only from the supplied
knowledge base. If the knowledge base does not cover the question, say you cannot help with that.
Keep every answer to one short sentence.
'''

TOPICS = [
    {
        "name": "booking-cancellation_policy",
        "content": (
            "USER QUESTIONS:\nWhat's your cancellation policy?\nCan I cancel my booking?\n"
            "What happens if I need to change my reservation?\nDo I get charged if I cancel?\n\n"
            "AVAILABLE RESPONSES:\n1. \"You can cancel or change a booking free of charge up to four hours "
            "before it starts. After that we charge twenty pounds per person.\""
        ),
    },
    {
        "name": "booking-large_groups",
        "content": (
            "USER QUESTIONS:\nCan I book for a large group?\nDo you take parties of more than eight?\n\n"
            "AVAILABLE RESPONSES:\n1. \"For parties of nine or more we take bookings by phone only, and we ask "
            "for a deposit.\""
        ),
    },
    {
        "name": "location-parking",
        "content": (
            "USER QUESTIONS:\nIs there parking?\nWhere can I park?\n\n"
            "AVAILABLE RESPONSES:\n1. \"There's a public car park on Bridge Street, two minutes' walk away. We "
            "don't have our own spaces.\""
        ),
    },
]

async with conversation(KNOWLEDGE_INSTRUCTIONS, knowledge=TOPICS) as conv:
    await conv.turn(wav="audio/kb_question.wav")
    await conv.turn(wav="audio/followup_parking.wav")
    await conv.turn(wav="audio/out_of_domain.wav")

### D. Text input, and what is on the wire

Text works too. A `conversation.item.create` carrying an `input_text` part is a caller turn like
any other, and the reply comes back the same way. Useful for testing prompts without audio, and
for channels that are text in the first place.

`conv.events` holds every server event exactly as it arrived. The last one is the `response.done`,
and the PolyAI fields on it are ordinary JSON keys, which is why an unmodified SDK can read them
from `model_extra` and never trips over them.

In [ ]:
async with conversation(KNOWLEDGE_INSTRUCTIONS, knowledge=TOPICS) as conv:
    await conv.turn(text="Do you take bookings for nine people?")
    last = conv.events[-1]

print(json.dumps(last, indent=2))

## 5. Delete the endpoint

The endpoint bills per hour until this runs. Stop the bridge first, then delete the endpoint, the
endpoint config and the model. Deleting the endpoint is the call that stops the charge.

In [ ]:
bridge.terminate()
bridge.wait(timeout=5)

sm.delete_endpoint(EndpointName=endpoint_name)
sm.delete_endpoint_config(EndpointConfigName=model_name)
sm.delete_model(ModelName=model_name)
print(f"deleted {endpoint_name}, {model_name}")

## 6. Unsubscribe from the listing (optional)

Before cancelling, make sure no deployable model created from this package remains in your
account: check the [models page](https://console.aws.amazon.com/sagemaker/home#/models) for
containers referencing it.

1. Open the **Machine Learning** tab of
   [your software subscriptions](https://aws.amazon.com/marketplace/ai/library?productType=ml).
2. Find the Dialog-RSN-1 listing and choose **Cancel Subscription**.

## 7. Going further

**Limits worth designing around.**

| | |
| --- | --- |
| Audio in | PCM16 mono, 16 kHz or 24 kHz. No G.711, Opus or stereo. |
| Output | Text only. Bring your own text-to-speech. |
| Context window | 32,000 tokens across instructions, tools, topics, history and audio. Old audio is compacted to its transcript automatically; a conversation long in text gets `context_length_exceeded`. |
| Session | 30 minutes per connection, a hard cap SageMaker enforces on the stream. Reconnect before it and replay what the next turn needs. An idle connection is closed after 5 minutes. |
| Turn detection | Server-side, not tunable. `create_response` is the one knob. |
| Language | English. |

**Building a voice agent.** The loop above is the whole protocol. A real agent adds text-to-speech
on each reply, stops playback when `input_audio_buffer.speech_started` arrives mid-reply (the model
has already decided the caller is interrupting), and validates tool arguments before acting on
them. Exact values such as dates, reference numbers and amounts are worth checking in your own
code.

**Skipping the bridge.** It is a pure pass-through. A client that calls
`InvokeEndpointWithBidirectionalStream` with `@aws-sdk/client-sagemaker-runtime-http2` and sends the
same JSON events as `PayloadPart` frames talks to the model directly, with no local process.

**Other clients.** Anything that speaks OpenAI Realtime over a WebSocket, in any language, connects
to the bridge the same way: base URL `ws://127.0.0.1:8079`, any API key, model `dialog-rsn-1`.